# Scroll ink sensitivity, stage 1: injection-recovery

Can today's 9 µm ink model see letters on an unread scroll? We measure the real ink signature on **PHerc. 139** (known text),
plant synthetic Greek letters (2.5 mm cap height) of controlled strength into real CT surface volumes, run the official model
(`scrollprize/ink_9um`) and count how many it recovers against a 5% false-positive threshold from empty control slots.

- **PHerc. 139** (calibration: the same scroll as the signature, alongside its real text)
- **PHerc. 800**: eligible for First Letters and the Grand Prize; its five published segments

Strength 1 = the measured PHerc. 139 ink signal. Settings: **GPU T4 x2**, Internet **on**. Best: **Save Version → Save & Run All**, about 1 hour.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Official code: ScrollPrize/villa, branch merge-ink-pipelines
!GIT_LFS_SKIP_SMUDGE=1 git clone -q --depth 1 --branch merge-ink-pipelines https://github.com/ScrollPrize/villa /kaggle/working/villa
!pip install -q uv imagecodecs
%cd /kaggle/working/villa/ink-detection
!uv sync -q 2>&1 | tail -5
!uv run python -c "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"

In [ ]:
# This project's code
!git clone -q --depth 1 https://github.com/Shakhtar-Sankur/ink-sensitivity /kaggle/working/ink-sensitivity
!cp /kaggle/working/ink-sensitivity/inject.py /kaggle/working/inject.py

In [ ]:
import glob
import json
import os
import shutil
import subprocess
import time

from huggingface_hub import hf_hub_download

W = "/kaggle/working"
ENV = "/kaggle/working/villa/ink-detection"  # the official environment (zarr 2, tifffile, imagecodecs)


def run(*cmd):
    r = subprocess.run(["uv", "run", "python", *cmd], cwd=ENV, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-2000:], r.stderr[-3000:])
        raise RuntimeError(f"failed: {cmd[:3]}")
    return r.stdout


print(run(f"{W}/inject.py", "signature", "--out", f"{W}/sig.json"))
VOLS = []
for seed in (0, 1, 2):
    for t in ["0139"] + [f"0800-{i}" for i in range(5)]:
        d = f"{W}/inj/{t}_s{seed}"
        os.makedirs(d, exist_ok=True)
        print(run(f"{W}/inject.py", "make", "--sig", f"{W}/sig.json", "--target", t, "--out", d, "--seed", str(seed)).strip())
        VOLS.append(d)

In [ ]:
# Official inference on every injected volume: two checkpoints, both depth directions
CKPTS = [hf_hub_download("scrollprize/ink_9um", f)
         for f in ("hybrid_3d2d-seed42/step-075000.pth", "hybrid_3d2d-seed43/step-020000.pth")]
DECODE = ("import glob, sys, numpy as np, tifffile; "
          "[np.save(p.rsplit('.', 1)[0] + '.npy', tifffile.imread(p)) for p in glob.glob(sys.argv[1] + '/ck*.tif*')]")
t_all = time.time()
for d in VOLS:
    t0 = time.time()
    ok = True
    for i, ck in enumerate(CKPTS):
        try:  # a volume normally takes 2-3 min; skip one that hangs instead of blocking the run
            r = subprocess.run(["uv", "run", "python", "-m", "koine_machines.inference.infer", f"{d}/volume.zarr", ck,
                                f"{d}/ck{i}.tif", "--direction", "both", "--batch-size", "16"],
                               cwd=ENV, capture_output=True, text=True, timeout=900)
        except subprocess.TimeoutExpired:
            print(f"timeout on {d}, checkpoint {i}: skipped")
            ok = False
            break
        if r.returncode != 0:
            print(f"inference failed on {d}: skipped\n", r.stdout[-1500:], r.stderr[-3000:])
            ok = False
            break
    if not ok:
        continue
    subprocess.run(["uv", "run", "python", "-c", DECODE, d], cwd=ENV, check=True)
    preds = sorted(glob.glob(f"{d}/ck*.npy"))
    print(run(f"{W}/inject.py", "score", "--dir", d, "--pred", *preds).splitlines()[0], f"({time.time() - t0:.0f} s)")
    shutil.rmtree(f"{d}/volume.zarr")  # keep the saved output small
print(f"all inference: {(time.time() - t_all) / 60:.1f} min")

In [ ]:
done = [d for d in VOLS if os.path.exists(f"{d}/scores.json")]
print(len(done), "of", len(VOLS), "volumes scored")
print(run(f"{W}/inject.py", "aggregate", "--dirs", *[d for d in done if "/0139_" in d],
          "--label", "PHerc. 139 (calibration)", "--out", f"{W}/agg_0139.json"))
print(run(f"{W}/inject.py", "aggregate", "--dirs", *[d for d in done if "/0800-" in d],
          "--label", "PHerc. 800 (eligible, unread)", "--out", f"{W}/agg_0800.json"))

In [ ]:
# Picture: model output on one PHerc. 800 segment, planted letters boxed with their strength
import numpy as np
from IPython.display import Image as Show
from IPython.display import display
from PIL import Image, ImageDraw

d = sorted(d for d in VOLS if "/0800-3_" in d and os.path.exists(f"{d}/scores.json"))[0]
meta = json.load(open(f"{d}/letters.json"))
pred = np.mean([np.load(p).astype(np.float32) for p in sorted(glob.glob(f"{d}/ck*.npy"))], axis=0) / 255.0
img = Image.fromarray((np.clip((pred - 0.25) / 0.5, 0, 1) * 255).astype(np.uint8)).convert("RGB")
dr = ImageDraw.Draw(img)
for L in meta["letters"]:
    dr.rectangle([L["x"], L["y"], L["x"] + L["w"], L["y"] + L["h"]], outline=(255, 60, 60), width=6)
    dr.text((L["x"], L["y"] - 40), f"{L['char']}  x{L['alpha']:g}", fill=(255, 200, 0))
for N in meta["null_slots"]:
    c = meta["cell_px"]
    dr.rectangle([N["x"], N["y"], N["x"] + c, N["y"] + c], outline=(80, 160, 255), width=3)
img.thumbnail((1600, 1600))
img.save(f"{W}/pherc0800_injection_example.png")
display(Show(filename=f"{W}/pherc0800_injection_example.png", width=1000))

In [ ]:
print("=== STAGE 1 SUMMARY (copy this) ===")
print("GPU:", subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
sig = json.load(open(f"{W}/sig.json"))
print("signature: peak", round(max(sig["d"], key=abs), 3), "sd; ink px", sig["ink_px"])
for name in ("agg_0139.json", "agg_0800.json"):
    print(name, json.dumps(json.load(open(f"{W}/{name}")), indent=None))